# v2ray / Xray — Proxy Protocols

v2ray/Xray support VMess, VLESS, Shadowsocks, SOCKS5 with various transport layers (TCP, WebSocket, gRPC, QUIC).

## Generate UUID for client ID

In [ ]:
import uuid
print("Generated UUID for v2ray client id:")
print(str(uuid.uuid4()))

## Build a VLESS + TLS config

In [ ]:
import json

def vless_server(port=443, client_id=None, cert="/etc/xray/certs/server.crt", key="/etc/xray/certs/server.key"):
    client_id = client_id or str(uuid.uuid4())
    return {
        "log": {"loglevel": "info"},
        "inbounds": [{
            "port": port,
            "protocol": "vless",
            "settings": {"clients": [{"id": client_id, "flow": "xtls-rprx-vision"}], "decryption": "none"},
            "streamSettings": {
                "network": "tcp",
                "security": "tls",
                "tlsSettings": {"certificates": [{"certificateFile": cert, "keyFile": key}]},
            },
            "tag": "vless-in",
        }],
        "outbounds": [{"protocol": "freedom", "tag": "direct"}],
    }

cid = str(uuid.uuid4())
cfg = vless_server(client_id=cid)
print(json.dumps(cfg, indent=2))
print(f"\nClient UUID: {cid}  (use this in the client config)")

## Build a VMess + WebSocket client config

In [ ]:
def vmess_client(server, port, client_id, ws_path="/vmess", local_socks=1080):
    return {
        "log": {"loglevel": "info"},
        "inbounds": [{"port": local_socks, "protocol": "socks",
                      "settings": {"auth": "noauth", "udp": True}}],
        "outbounds": [{
            "protocol": "vmess",
            "settings": {"vnext": [{"address": server, "port": port,
                "users": [{"id": client_id, "alterId": 0, "security": "auto"}]}]},
            "streamSettings": {"network": "ws", "wsSettings": {"path": ws_path}},
            "tag": "vmess-out",
        }, {"protocol": "freedom", "tag": "direct"}],
        "routing": {"rules": [{"type": "field", "ip": ["geoip:private"], "outboundTag": "direct"}]},
    }

print(json.dumps(vmess_client("proxy.example.com", 10086, str(uuid.uuid4())), indent=2))